In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.float_format", "{:.3f}".format)

from scipy.stats import norm
from scipy.linalg import eig

import statsmodels.api as sm
import statsmodels.formula.api as smf

import matplotlib.pyplot as plt

# Reproducibility
np.random.seed(53241986)

## Synthetic Data Generation and Inference Workflow

In this notebook, we generate synthetic data from an individual-based simulation (IBM) of a species’ population cycle, using fixed true parameters for the vital-rate models. We assume a time-homogeneous demographic process, meaning that the kernel function and vital-rate functions do not depend on time.

This simulation framework follows the IBM approach used for monocarp species in Ellner and Rees (2016), based on the code here:
https://github.com/ipmbook/first-edition/blob/master/Rcode/c2/Monocarp%20Calculations.R

### Vital rates

We model five vital rates:

1. Growth(z1, z) ~ Normal(f(size), sigma)
2. Flowering ~ Bernoulli(sigmoid(f(size)))
3. Survival ~ Bernoulli(sigmoid(f(size)))
4. Seed production ~ Poisson(exp(f(size)))
5. Recruit size ~ N(f(size), sigma)

### Simulation procedure

* We simulate the IBM for 100 years, starting from an initial population of 250 plants.
* The first 10 years are discarded, since the initial population is arbitrary and has not yet reached the stable population structure generated by the life cycle process.
* Each row of the resulting dataset represents one individual observed over one time interval.
* We then subsample the population to mimic random sampling of individuals.

This subsampling is valid because the demographic process is assumed to be time invariant. Under this assumption, observing an individual over years 4–5 is equivalent to observing another individual over years 67–68.

### Inference

Using the subsampled data as if it were real observations, we infer the true model parameters.

The vital-rate functions are fitted using generalized linear models (GLMs).

### IPM construction

Using both the true parameters and the estimated parameters, we construct integral projection models (IPMs). The integral is discretized using the midpoint method, which converts the kernel function into a kernel matrix whose entries represent transition probabilities. These probabilities are obtained from the fitted model parameters.

Finally, we compute population-level statistics for both the true and estimated kernel matrices and compare the results.


In [46]:
# True model parameters
m_par_true = {
    "surv.int": -0.65, #intercept
    "surv.z": 0.75,    #slope

    "flow.int": -18.0,
    "flow.z": 6.9,

    "grow.int": 0.96,
    "grow.z": 0.59,
    "grow.sd": 0.67,

    "rcsz.int": -0.08,
    "rcsz.sd": 0.76,

    "seed.int": 1.00,
    "seed.z": 2.20,

    "p.r": 0.007
}

In [47]:
# Growth kernel
def G_z1z(z1, z, m_par):
    """
    Growth probability density

    args:
    zi: float or numpy array [size at time t+1]
    z: float or numpy array [size at time t]
    m_par: dict [dictionary containing demographic parameters]

    returns:
    float or numpy array [probability density of growing from z to z1]
    """
    mean_size_next_year = (
        m_par["grow.int"] + m_par["grow.z"] * z
    )

    growth_sd = m_par["grow.sd"]

    probability_density = norm.pdf(
        z1,
        loc=mean_size_next_year,
        scale=growth_sd
    )

    return probability_density


In [4]:
def sigmoid(x):
    return 1/(1 + np.exp(-x))
    
# Survival Kernel
def s_z(z, m_par):
    """
    Survival Probability as a Function of Size
    """

    linear_predictor = (
        m_par["surv.int"] + m_par["surv.z"] * z
    )

    survival_probability = sigmoid(linear_predictor)
    return survival_probability

In [5]:
# Flowering Probability
def p_bz(z, m_par):
    """
    Probability that a plant flowers.
    """

    linear_predictor = (
        m_par["flow.int"] +
        m_par["flow.z"] * z
    )

    flowering_probability = sigmoid(linear_predictor)

    return flowering_probability

In [6]:
# Seed Production [Poisson mean function]
def b_z(z, m_par):
    """
    Expected number of seeds produced by a flowering plant.
    """
    linear_predictor = m_par["seed.int"] + m_par["seed.z"] * z

    expected_seed_number = np.exp(linear_predictor)

    return expected_seed_number

In [7]:
# Probability density of recruit size
def c_0z1(z1, m_par):
    """
    Probability density of recruit size.
    """

    recruit_mean_size = m_par["rcsz.int"]

    recruit_standard_deviation = m_par["rcsz.sd"]

    recruit_density = norm.pdf(
        z1,
        loc=recruit_mean_size,
        scale=recruit_standard_deviation
    )

    return recruit_density

### IBM Simulations


In [53]:
init_pop_size = 250
n_years = 100

plant_sizes = np.random.normal(
    loc = m_par_true['rcsz.int'],
    scale = m_par_true['rcsz.sd'],
    size = init_pop_size
 )

plant_ages = np.zeros(init_pop_size, dtype = int)
population_size_history = [len(plant_sizes)]

mean_size_history = [plant_sizes.mean()]

mean_age_history = [plant_ages.mean()]
simulation_records = []
recruits_per_year = []

for year in range(1, n_years + 1):
    current_population_size = len(plant_sizes)
    print(f"Year {year}:Current Population Size = {current_population_size}")

    flowered = np.random.binomial(
        n = 1,
        p = p_bz(plant_sizes, m_par_true)
    )

    number_flowering = flowered.sum()
    seed_counts = np.full(
        shape = len(plant_sizes),
        fill_value = np.nan
    )

    seed_counts[flowered == 1] = np.random.poisson(
        lam = b_z(
            plant_sizes[flowered == 1],
            m_par_true
        ) 
    )

    total_seeds = np.nansum(seed_counts)

    if number_flowering == 0:
        number_recruits = 0
    else:
        # Each seed has some probability of becoming a new recruit
        number_recruits = np.random.binomial(
            n = int(total_seeds),
            p = m_par_true["p.r"]
        )
    recruits_per_year.append(number_recruits)

    recruit_sizes = np.random.normal(
        loc = m_par_true["rcsz.int"],
        scale = m_par_true["rcsz.sd"],
        size = number_recruits
    )

    survived = np.full(
        shape = len(plant_sizes),
        fill_value = np.nan
        )

    non_flowering = (flowered == 0)
    survival_probabilities = s_z(
        plant_sizes[non_flowering],
        m_par_true
    )

    survived[non_flowering] = np.random.binomial(
        n = 1,
        p = survival_probabilities
    )

    still_alive = ((flowered == 0) & (survived ==1))
    expected_next_size = (
        m_par_true["grow.int"] + m_par_true["grow.z"] * plant_sizes[still_alive]
    )

    grown_sizes = np.random.normal(
        loc = expected_next_size,
        scale = m_par_true["grow.sd"]
    )

    next_year_size = np.full(len(plant_sizes), np.nan)
    next_year_size[still_alive] = grown_sizes

    for i in range(len(plant_sizes)):
        simulation_records.append({
            "z": plant_sizes[i],
            "Repr": flowered[i],
            "Seeds": seed_counts[i],
            "Surv": survived[i],
            "z1": next_year_size[i],
            "age": plant_ages[i],
            "alive": still_alive[i],
            "yr": year
        })

    plant_sizes = np.concatenate([recruit_sizes, grown_sizes])

    plant_ages = np.concatenate([
        np.zeros(number_recruits, dtype=int),
        plant_ages[still_alive] + 1
    ])

sim_data = pd.DataFrame(simulation_records)

Year 1:Current Population Size = 250
Year 2:Current Population Size = 91
Year 3:Current Population Size = 131
Year 4:Current Population Size = 145
Year 5:Current Population Size = 112
Year 6:Current Population Size = 141
Year 7:Current Population Size = 208
Year 8:Current Population Size = 177
Year 9:Current Population Size = 156
Year 10:Current Population Size = 121
Year 11:Current Population Size = 224
Year 12:Current Population Size = 242
Year 13:Current Population Size = 286
Year 14:Current Population Size = 257
Year 15:Current Population Size = 191
Year 16:Current Population Size = 265
Year 17:Current Population Size = 344
Year 18:Current Population Size = 355
Year 19:Current Population Size = 327
Year 20:Current Population Size = 368
Year 21:Current Population Size = 542
Year 22:Current Population Size = 593
Year 23:Current Population Size = 432
Year 24:Current Population Size = 417
Year 25:Current Population Size = 861
Year 26:Current Population Size = 719
Year 27:Current Popula

In [9]:
sim_data.describe()

,z,Repr,Seeds,Surv,z1,age,yr
count,4829455.000,4829455.000,180792.000,4648663.000,1962521.000,4829455.000,4829455.000
mean,0.492,0.037,2535.114,0.422,1.426,0.728,86.108
std,1.087,0.190,4762.102,0.494,0.874,1.237,14.088
min,-4.045,0.000,15.000,0.000,-2.621,0.000,1.000
25%,-0.295,0.000,844.000,0.000,0.830,0.000,80.000
50%,0.390,0.000,1409.000,0.000,1.430,0.000,90.000
75%,1.229,0.000,2597.000,1.000,2.027,1.000,96.000
max,5.519,1.000,510863.000,1.000,5.519,21.000,100.000


In [50]:
sim_data = sim_data[sim_data["yr"] > 10].copy()

sim_data["yr"] = sim_data["yr"] - 10

In [52]:
dataset_sizes = [250,500,750,1000,2000]
dataset_dict = {}

for n in dataset_sizes:
    data_sample = sim_data.sample(
    n=n,
    replace=False,
    random_state=53241986
    )
    data_sample["z_classes"] = pd.cut(
        sim_data["z"],
        bins = 20
    )
    dataset_dict[f"sim_data{n}"] = data_sample
    data_sample.to_csv(f"sim_data{n}.csv", index=False)

In [12]:
sim_data = dataset_dict["sim_data250"]

In [54]:
sim_data.head()

,z,Repr,Seeds,Surv,z1,age,alive,yr
0,0.097,0,NaN,0.000,NaN,0,False,1
1,-0.832,0,NaN,0.000,NaN,0,False,1
2,-0.756,0,NaN,1.000,0.282,0,True,1
3,-0.130,0,NaN,0.000,NaN,0,False,1
4,1.430,0,NaN,1.000,2.458,0,True,1


In [ ]:
sim_data = sim_data.sample(
    n=250,
    replace=False,
    random_state=53241986
)

In [14]:
sim_data["z_classes"] = pd.cut(
    sim_data["z"],
    bins=20
)

sim_data = sim_data.sort_values("z")

In [15]:
growth_data = sim_data.dropna(subset = ["z1"]).copy()

In [16]:
print(len(growth_data))

97


In [17]:
growth_model = smf.ols(
    formula="z1 ~ z",
    data=growth_data
).fit()

print(growth_model.summary())

                            OLS Regression Results                            
Dep. Variable:                     z1   R-squared:                       0.267
Model:                            OLS   Adj. R-squared:                  0.259
Method:                 Least Squares   F-statistic:                     34.53
Date:                Thu, 09 Jul 2026   Prob (F-statistic):           6.19e-08
Time:                        18:03:02   Log-Likelihood:                -88.474
No. Observations:                  97   AIC:                             180.9
Df Residuals:                      95   BIC:                             186.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.0149      0.082     12.368      0.0

In [18]:
growth_intercept = growth_model.params["Intercept"]
growth_slope = growth_model.params["z"]
growth_sd = np.sqrt(growth_model.scale)

In [19]:
flowering_model = smf.glm(
    formula="Repr ~ z",
    data=sim_data,
    family=sm.families.Binomial()
).fit()
flowering_intercept = flowering_model.params["Intercept"]
flowering_slope = flowering_model.params["z"]
print(flowering_model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                   Repr   No. Observations:                  250
Model:                            GLM   Df Residuals:                      248
Model Family:                Binomial   Df Model:                            1
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -17.426
Date:                Thu, 09 Jul 2026   Deviance:                       34.853
Time:                        18:03:02   Pearson chi2:                     37.9
No. Iterations:                    10   Pseudo R-squ. (CS):             0.2361
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept    -11.1483      2.747     -4.058      0.0

In [20]:
# Keep only non-flowering plants
survival_data = sim_data[sim_data["Repr"] == 0].copy()

# Fit logistic regression
survival_model = smf.glm(
    formula="Surv ~ z",
    data=survival_data,
    family=sm.families.Binomial()
).fit()

print(survival_model.summary())

# Extract coefficients
survival_intercept = survival_model.params["Intercept"]
survival_slope = survival_model.params["z"]

                 Generalized Linear Model Regression Results                  
Dep. Variable:                   Surv   No. Observations:                  237
Model:                            GLM   Df Residuals:                      235
Model Family:                Binomial   Df Model:                            1
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -142.56
Date:                Thu, 09 Jul 2026   Deviance:                       285.12
Time:                        18:03:02   Pearson chi2:                     241.
No. Iterations:                     4   Pseudo R-squ. (CS):             0.1394
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     -0.7501      0.163     -4.609      0.0

In [21]:
# Keep only flowering plants
seed_data = sim_data[sim_data["Repr"] == 1].copy()

# Fit Poisson regression
seed_model = smf.glm(
    formula="Seeds ~ z",
    data=seed_data,
    family=sm.families.Poisson()
).fit()

print(seed_model.summary())

# Extract coefficients
seed_intercept = seed_model.params["Intercept"]
seed_slope = seed_model.params["z"]

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  Seeds   No. Observations:                   13
Model:                            GLM   Df Residuals:                       11
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -61.356
Date:                Thu, 09 Jul 2026   Deviance:                       10.213
Time:                        18:03:02   Pearson chi2:                     10.2
No. Iterations:                     5   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.9783      0.065     15.114      0.0

In [22]:
# Recruits have age 0
recruit_data = sim_data[sim_data["age"] == 0].copy()

recruit_mean = recruit_data["z"].mean()
recruit_sd = recruit_data["z"].std()

print(recruit_mean)
print(recruit_sd)

-0.03752209747565997
0.7845370578093546


In [23]:
sim_data

,z,Repr,Seeds,Surv,z1,age,alive,yr,z_classes
821949,-2.232,0,NaN,1.000,1.420,0,True,65,"(-2.238, -1.954]"
3036181,-1.882,0,NaN,0.000,NaN,0,False,84,"(-1.954, -1.676]"
1604429,-1.786,0,NaN,0.000,NaN,0,False,75,"(-1.954, -1.676]"
436352,-1.725,0,NaN,0.000,NaN,0,False,56,"(-1.954, -1.676]"
2783584,-1.644,0,NaN,0.000,NaN,0,False,83,"(-1.676, -1.398]"
...,...,...,...,...,...,...,...,...,...
3905183,2.775,0,NaN,1.000,3.473,3,True,87,"(2.773, 3.051]"
4150001,2.830,1,1380.000,NaN,NaN,2,False,88,"(2.773, 3.051]"
4799239,3.093,1,2479.000,NaN,NaN,2,False,90,"(3.051, 3.329]"
1796315,3.240,1,3475.000,NaN,NaN,1,False,76,"(3.051, 3.329]"


In [34]:
p_r_est = (
    (sim_data["age"] == 0).sum()
    /
    np.nansum(sim_data["Seeds"])
)

print(p_r_est)

0.00928757503681051


In [35]:
m_par_est = {
    "surv.int": survival_intercept,
    "surv.z": survival_slope,

    "flow.int": flowering_intercept,
    "flow.z": flowering_slope,

    "grow.int": growth_intercept,
    "grow.z": growth_slope,
    "grow.sd": growth_sd,

    "rcsz.int": recruit_mean,
    "rcsz.sd": recruit_sd,

    "seed.int": seed_intercept,
    "seed.z": seed_slope,

    "p.r": p_r_est
}

In [36]:
def P_z1z(z1, z, m_par):
    """
    Survival-growth kernel.
    """

    return (
        (1 - p_bz(z, m_par))
        * s_z(z, m_par)
        * G_z1z(z1, z, m_par)
    )

In [37]:
def F_z1z(z1, z, m_par):
    """
    Fecundity kernel.
    """

    return (
        p_bz(z, m_par)
        * b_z(z, m_par)
        * m_par["p.r"]
        * c_0z1(z1, m_par)
    )

In [38]:
def mk_K(n_mesh_points, m_par, lower_size, upper_size):

    # Width of each mesh interval
    mesh_width = (upper_size - lower_size) / n_mesh_points

    # Midpoints of each interval
    mesh_points = (
        lower_size
        + (np.arange(n_mesh_points) + 0.5)
        * mesh_width
    )

    # Evaluate kernels
    z1_grid, z_grid = np.meshgrid(
        mesh_points,
        mesh_points,
        indexing="ij"
    )

    P = (           # P is a (n_mesh_points x n_mesh_points) matrix of transition probabilities
        mesh_width
        * P_z1z(z1_grid, z_grid, m_par)
    )

    F = (           # P is a (n_mesh_points x n_mesh_points) matrix of fecundity probabilities
        mesh_width
        * F_z1z(z1_grid, z_grid, m_par)
    )

    K = P + F

    return {
        "K": K,
        "P": P,
        "F": F,
        "mesh_points": mesh_points
    }

In [39]:
n_big_matrix = 250

IPM_true = mk_K(
    n_mesh_points=n_big_matrix,
    m_par=m_par_true,
    lower_size=-2.65,
    upper_size=4.5
)

IPM_est = mk_K(
    n_mesh_points=n_big_matrix,
    m_par=m_par_est,
    lower_size=-2.65,
    upper_size=4.5
)

In [40]:
eigenvalues_true, eigenvectors_true = np.linalg.eig(IPM_true["K"])

lambda_true = np.max(eigenvalues_true.real)

print(lambda_true)

1.059306809536006


In [41]:
eigenvalues_est, eigenvectors_est = np.linalg.eig(IPM_est["K"])

lambda_est = np.max(eigenvalues_est.real)

print(lambda_est)

0.8578282976118394


In [42]:
dominant_index = np.argmax(eigenvalues_est.real)

stable_distribution = (
    eigenvectors_est[:, dominant_index].real
)

stable_distribution /= stable_distribution.sum()

In [43]:
mean_size = np.sum(
    stable_distribution
    * IPM_est["mesh_points"]
)

print(mean_size)

0.7046919298898581
